<a href="https://colab.research.google.com/github/javisnes/E-commerce-SQL/blob/main/E_commerce_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import sqlite3
import pandas as pd
import random
import numpy as np
from datetime import datetime, timedelta

# 1. Crear conexión a la base de datos local
conn = sqlite3.connect('ecommerce_sales.db')

# 2. Generar datos simulados
random.seed(42)
np.random.seed(42)

# Tabla Clientes
clientes = pd.DataFrame({
    'ID_Cliente': range(1, 101),
    'Pais': [random.choice(['Mexico', 'Colombia', 'Argentina', 'España', 'Chile']) for _ in range(100)],
    'Suscripcion': [random.choice(['Premium', 'Gratis']) for _ in range(100)]
})

# Tabla Productos
productos = pd.DataFrame({
    'ID_Producto': range(200, 210),
    'Categoria': ['Electronica', 'Ropa', 'Hogar', 'Deportes', 'Libros'] * 2,
    'Precio': np.random.uniform(15.0, 500.0, 10).round(2)
})

# Tabla Ventas
fechas = [datetime(2023, 1, 1) + timedelta(days=random.randint(0, 365)) for _ in range(500)]
ventas = pd.DataFrame({
    'ID_Venta': range(1000, 1500),
    'ID_Cliente': [random.choice(clientes['ID_Cliente']) for _ in range(500)],
    'ID_Producto': [random.choice(productos['ID_Producto']) for _ in range(500)],
    'Cantidad': [random.randint(1, 5) for _ in range(500)],
    'Fecha': [f.strftime('%Y-%m-%d') for f in fechas]
})

# 3. Insertar DataFrames en SQLite
clientes.to_sql('Clientes', conn, index=False, if_exists='replace')
productos.to_sql('Productos', conn, index=False, if_exists='replace')
ventas.to_sql('Ventas', conn, index=False, if_exists='replace')

print("✅ Base de datos 'ecommerce_sales.db' creada exitosamente con 3 tablas.")

✅ Base de datos 'ecommerce_sales.db' creada exitosamente con 3 tablas.


In [8]:
# Cargar la extensión SQL de Colab
%load_ext sql

# Conectar a la base de datos
%sql sqlite:///ecommerce_sales.db

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


### 🔍 1. Exploración Básica: Conociendo los Datos
**Objetivo:** Dar un primer vistazo a la información cruda que tenemos en nuestra tabla de ventas antes de hacer cálculos complejos.

**¿Cómo funciona esta consulta?**
* `SELECT *`: El asterisco `*` es un comodín en SQL que significa "Tráeme absolutamente TODAS las columnas" de la tabla.
* `FROM Ventas`: Le indica al motor de base de datos en qué "carpeta" o tabla debe buscar.
* `LIMIT 5`: Es una medida de seguridad y limpieza. Le dice a SQL "Solo muéstrame los primeros 5 registros". Si no lo ponemos, SQL intentaría imprimir en pantalla las 500 ventas, lo cual haría el análisis muy lento y desordenado.

In [13]:
# 1. Hago la consulta SQL pura dentro de las comillas triples
consulta = """
SELECT * FROM Ventas
LIMIT 5;
"""

# 2. Le pido a Pandas que ejecute ese SQL en la coneccion (conn)
tabla_resultado = pd.read_sql_query(consulta, conn)

# 3. Muestro el resultado
display(tabla_resultado)

,ID_Venta,ID_Cliente,ID_Producto,Cantidad,Fecha
0,1000,74,202,4,2023-07-26
1,1001,87,204,5,2023-06-23
2,1002,49,200,2,2023-02-25
3,1003,51,200,1,2023-05-08
4,1004,92,205,4,2023-04-09


### 🤝 2. Uniendo Tablas (INNER JOIN): Enriqueciendo la Información
**Objetivo:** Las bases de datos reales están normalizadas, lo que significa que la tabla de `Ventas` solo guarda números (IDs) para ahorrar espacio. Aquí cruzaremos las ventas con el catálogo de `Productos` para obtener un ticket de compra legible para humanos.

**Análisis del código paso a paso:**
* `SELECT ...`: A diferencia del asterisco, aquí listamos exactamente las columnas que queremos en nuestro reporte final (ID_Venta, Fecha, Categoria, Precio, Cantidad).
* `(Productos.Precio * Ventas.Cantidad) AS Total_Venta`: SQL también es una calculadora. Aquí le pedimos que multiplique dos columnas en tiempo real y que el resultado lo guarde en una **nueva columna virtual** que bautizamos como `Total_Venta` usando el comando `AS`.
* `FROM Ventas`: Partimos de nuestra tabla de tickets.
* `JOIN Productos`: Le decimos que traiga la tabla del catálogo para unirla.
* `ON Ventas.ID_Producto = Productos.ID_Producto`: **Este es el puente.** Le explicamos a SQL cuál es la regla de coincidencia: *"Si el ID_Producto en el ticket es 205, búscame el 205 en el catálogo y pega sus datos aquí"*.

In [14]:
consulta_join = """
SELECT
    Ventas.ID_Venta,
    Ventas.Fecha,
    Productos.Categoria,
    Productos.Precio,
    Ventas.Cantidad,
    (Productos.Precio * Ventas.Cantidad) AS Total_Venta
FROM Ventas
JOIN Productos ON Ventas.ID_Producto = Productos.ID_Producto
LIMIT 10;
"""

ventas_detalladas = pd.read_sql_query(consulta_join, conn)
display(ventas_detalladas)

,ID_Venta,Fecha,Categoria,Precio,Cantidad,Total_Venta
0,1000,2023-07-26,Hogar,370.02,4,1480.08
1,1001,2023-06-23,Libros,90.67,5,453.35
2,1002,2023-02-25,Electronica,196.65,2,393.30
3,1003,2023-05-08,Electronica,196.65,1,196.65
4,1004,2023-04-09,Electronica,90.66,4,362.64
5,1005,2023-04-08,Electronica,196.65,1,196.65
6,1006,2023-10-02,Libros,90.67,3,272.01
7,1007,2023-08-18,Electronica,90.66,1,90.66
8,1008,2023-03-13,Electronica,90.66,5,453.30
9,1009,2023-08-05,Ropa,43.17,2,86.34


### 📊 3. Agrupación y Agregación (GROUP BY): Resumen de Negocios
**Objetivo:** Responder a la pregunta de negocio: *"¿Qué categoría de productos nos está generando más ingresos?"*. Para esto, necesitamos agrupar cientos de ventas individuales en un resumen.

**Análisis del código paso a paso:**
* `SUM(...) AS Dinero_Total`: Usamos la función de agregación `SUM`. SQL va a tomar todas las multiplicaciones de (Precio * Cantidad) y las va a sumar todas juntas para darnos el Gran Total.
* `GROUP BY Productos.Categoria`: Este es el núcleo del análisis. Le dice a SQL: *"No me muestres cada venta. Mejor haz montoncitos separados por cada Categoría (Hogar, Ropa, etc.) y aplícale la suma a cada montón por separado"*.
* `ORDER BY Dinero_Total DESC`: Para que el reporte esté listo para la gerencia, le pedimos que ordene la tabla resultante basándose en la columna del dinero, de mayor a menor (`DESC` significa Descendente). Así, la categoría estrella queda en el primer renglón.

In [15]:
consulta_agrupada = """
SELECT
    Productos.Categoria,
    SUM(Productos.Precio * Ventas.Cantidad) AS Dinero_Total
FROM Ventas
JOIN Productos ON Ventas.ID_Producto = Productos.ID_Producto
GROUP BY Productos.Categoria
ORDER BY Dinero_Total DESC;
"""

# Le pedimos a Pandas que ejecute la consulta y nos muestre el resultado
ventas_por_categoria = pd.read_sql_query(consulta_agrupada, conn)
display(ventas_por_categoria)

,Categoria,Dinero_Total
0,Hogar,109671.10
1,Ropa,96674.25
2,Libros,79653.23
3,Deportes,74967.83
4,Electronica,42156.60
